In [1]:
from pandas import read_csv, read_excel, DataFrame
from rdkit import Chem
from rdkit.Chem import Fragments

In [2]:
filename = "./Sources/SolventsDatasets/dataset_CAS_CID_SMILES_Tm_Tb_Sources.csv"
data : DataFrame = read_csv(filename, delimiter=",")

`RDKit - Functional Groups`

In [3]:
def get_fragment_count(smiles, fragment_func):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return fragment_func(mol)

def count_matches(smiles, patt):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    return len(mol.GetSubstructMatches(patt))

In [4]:
fragment_functions = {
    'Solvent_halide': Fragments.fr_halogen,
    'Solvent_ketone': Fragments.fr_ketone,
    'Solvent_ester': Fragments.fr_ester,
    'Solvent_nitrile': Fragments.fr_nitrile,
    'Solvent_benzene': Fragments.fr_benzene,
    'Solvent_carboxylic_acid': Fragments.fr_COO2,
    'Solvent_amide': Fragments.fr_amide,
    'Solvent_pyridine': Fragments.fr_pyridine,
    'Solvent_sulfone': Fragments.fr_sulfone,
    'Solvent_ether' : Fragments.fr_ether
    
}

for col_name, frag_func in fragment_functions.items():
    data[col_name] = data['SMILES'].apply(lambda x: get_fragment_count(x, frag_func))

def count_total_alcohol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    al_oh = Fragments.fr_Al_OH(mol)
    ar_oh = Fragments.fr_Ar_OH(mol)
    return al_oh + ar_oh

data['Solvent_Alcohol'] = data['SMILES'].apply(count_total_alcohol)

`SMARTS - Functional Groups`

In [5]:
pattern_individual_smarts = {
    'Solvent_sulfoxide': '[#16D3](=[#8])([#6])[#6]',    
    'Solvent_C=C-alq':   '[#6]=[#6]',                
    'Solvent_carbonate': '[#6X3](=O)(O[!#1])O[!#1]',    
}
smarts_patterns = {n: Chem.MolFromSmarts(p) for n, p in pattern_individual_smarts.items()}

def is_water(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    return int(Chem.MolToSmiles(mol) == 'O')

for col, pattern in smarts_patterns.items():
    data[col] = data['SMILES'].apply(lambda s, p=pattern: count_matches(s, p))

data['Solvent_water'] = data['SMILES'].apply(is_water)

Carbonyl and nitrile group satisfy the [C;H0;!a] pattern but are already accounted for by the corresponding functional-group descriptors and are therefore substracted to prevent double counting. Amide carbonyls are retained, as they are not represented by a carbonyl-carbon descriptor in this feature set.

In [6]:
carbon_individual_smarts = {
    'Solvent_CH3': '[C;H3;!a]',
    'Solvent_CH2': '[C;H2;!a]',
    'Solvent_CH':  '[C;H1;!a]',
    'Solvent_C':   '[C;H0;!a]',   
}
carbon_patterns = {n: Chem.MolFromSmarts(s) for n, s in carbon_individual_smarts.items()}

exclusion_smarts = {
    'carbonyl':   '[CX3]=O',           
    'amide_carbonyl': '[CX3](=O)[NX3]',    
    'nitrile':  'C#N',              
}
exclusion_patterns = {n: Chem.MolFromSmarts(s) for n, s in exclusion_smarts.items()}

def count_solvent_C(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    targets  = {m[0] for m in mol.GetSubstructMatches(carbon_patterns['Solvent_C'])}
    all_carbonyl  = {m[0] for m in mol.GetSubstructMatches(exclusion_patterns['carbonyl'])}
    amide_carbonyl = {m[0] for m in mol.GetSubstructMatches(exclusion_patterns['amide_carbonyl'])}
    excluded = all_carbonyl - amide_carbonyl
    for m in mol.GetSubstructMatches(exclusion_patterns['nitrile']):
        excluded.update(m)
    return len(targets - excluded)

for col in ['Solvent_CH3', 'Solvent_CH2', 'Solvent_CH']:
    pattern = carbon_patterns[col]
    data[col] = data['SMILES'].apply(lambda s, p=pattern: count_matches(s, p))

data['Solvent_C'] = data['SMILES'].apply(count_solvent_C)

The structure of the compounds and their assigned functional groups were visually assessed. The following code blocks apply corrections to specific compounds where the automated descriptor assignment did not match the expected functional-group count. In addition, compounds containing functional groups outside the scope of the descriptor set were removed from the database manually.

In [7]:
def update_solvent_value(df, cid, column, new_value):
    df.loc[df["CID"] == cid, column] = new_value
    return True

In [8]:
solvent_ketone = {
    3611: 1,
    7167: 1,
    77828: 1,
    89781: 1,
}

for cid, value in solvent_ketone.items():
    update_solvent_value(data, cid, "Solvent_ketone", value)

In [9]:
solvent_ester = {
    7167: 1,
    77828: 1,
    89781: 1,
    6436487: 2
}

for cid, value in solvent_ester.items():
    update_solvent_value(data, cid, "Solvent_ester", value)

In [10]:
solvent_amide = {78343: 1}

for cid, value in solvent_amide .items():
    update_solvent_value(data, cid, "Solvent_amide", value)

In [11]:
solvent_pyridine = {78343: 0}

for cid, value in solvent_pyridine .items():
    update_solvent_value(data, cid, "Solvent_pyridine", value)

In [12]:
solvent_sulfoxide = {20481: 1}

for cid, value in solvent_sulfoxide .items():
    update_solvent_value(data, cid, "Solvent_sulfoxide", value)

In [13]:
solvent_C = {
    3515: 5,
    3611: 1,
    6436487: 1
}

for cid, value in solvent_C.items():
    update_solvent_value(data, cid, "Solvent_C", value)

In [14]:
solvent_CH = {
    284: 0,
    3515: 6,
    3611: 5,
    78343: 2,
    6436487: 3
}

for cid, value in solvent_CH.items():
    update_solvent_value(data, cid, "Solvent_CH", value)

In [15]:
solvent_sulfone = {
    4156: 1,
    5937: 2,
    6163: 2,
    6497: 2,
    6645: 1,
    14264: 1,
    15411: 1
}

for cid, value in solvent_sulfone.items():
    update_solvent_value(data, cid, "Solvent_sulfone", value)

In [16]:
solvent_CC_alq = {
    78343: 1,
    136075: 0,
    136327: 0,
    138779: 1,
    141735: 0,
    143960: 0,
    144711: 2,
    145625: 0,
    6436487: 2,
}

for cid, value in solvent_CC_alq.items():
    update_solvent_value(data, cid, "Solvent_C=C-alq", value)

In [17]:
data['Solvent_ether'] = (data['Solvent_ester'] - data['Solvent_ether']).abs()
data['Solvent_ether'] = data['Solvent_ether'] - 2 * data['Solvent_carbonate']

In [18]:
solvent_ether = {
    5055: 1, 5541: 3, 14264: 1, 4133: 1, 660: 1, 2345: 1,
    15411: 1, 6436487: 0, 281853: 1, 6490: 1, 77828: 1,
    20481: 2, 2893: 1, 5937: 2, 2365: 1, 6050: 3, 701: 1,
    6163: 2, 4156: 1, 6497: 2, 6505: 4, 6506: 3, 6507: 3,
    6532: 1, 6631: 1, 6645: 1, 6658: 1, 6781: 2, 6782: 2,
    3026: 2, 6786: 2, 6788: 2, 2347: 2, 6819: 3, 6873: 1,
    6880: 1, 6910: 2, 7136: 0, 7150: 1, 7165: 1, 7167: 1,
    7170: 1, 7193: 1, 7217: 0, 7234: 1, 7268: 2, 7274: 1,
    7294: 1, 7302: 1, 7334: 1, 7335: 1, 7336: 1, 7342: 1,
}

for cid, value in solvent_ether.items():
    update_solvent_value(data, cid, "Solvent_ether", value)

`Remove CIDS`

In [19]:
with open("./Sources/SolventsDatasets/cids_remove.txt", "r") as f:
    remove_cids = [int(line.strip()) for line in f if line.strip()]

data = data[~data["CID"].isin(remove_cids)].copy()

print(f"Removed {len(remove_cids)} CIDs")
print(f"Remaining rows: {len(data)}")

Removed 707 CIDs
Remaining rows: 9496


In [23]:
data = data.rename(columns={"Tm": "Temperature Melting (K)", "Tb": "Temperature Boiling  (K)"})

In [ ]:
#data.to_csv("./Sources/SolventsDatasets/9496_SolventsPubChem.csv", index = False)

In [ ]:
data['CAS'] = data['CAS'].astype(str)
#data.to_excel("./Sources/SolventsDatasets/9496_SolventsPubChem.xlsx", index = False)